# **Step 10: Final Evaluation**

Now compare all three stages:
1.   Base model
2.   Instruction fine-tuned model
3.   DPO-aligned model

In [ ]:
!pip -q install groq reportlab huggingface_hub

In [ ]:
# from kaggle_secrets import UserSecretsClient
# from groq import Groq

# user_secrets = UserSecretsClient()
# GROQ_API_KEY = userdata.get("GROQ_API_KEY")
# groq_client = Groq(api_key=GROQ_API_KEY)
# HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
from google.colab import userdata
from groq import Groq

HF_TOKEN = userdata.get("HF_TOKEN")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

In [ ]:
import os
import json
import re
import time
from datetime import datetime
from google.colab import userdata
from groq import Groq
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

In [ ]:
# =====================================================================
# 1. INITIALIZE ACTIVE PRODUCTION INFRASTRUCTURE
# =====================================================================
print("🔧 Fetching Groq credentials and initializing client...")

from huggingface_hub import (
    login,
    HfApi,
)

HF_REPO_NAME = "Pzazz55"

HF_REPOS = {
    "dataset": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-dataset",
    "stage1": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage1",
    "stage2": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage2",
    "stage3": f"{HF_REPO_NAME}/insurance-ai-assistant-finetuning-stage3",
}

# CONFIGURED_MODEL = "llama-3.3-70b-versatile"
CONFIGURED_MODEL = "llama-3.1-8b-instant"


MAX_ANSWER_CHARS = 1500

🔧 Fetching Groq credentials and initializing client...


In [ ]:
# =====================================================
# Download Latest Stage 3 Report
# =====================================================

from huggingface_hub import hf_hub_download, HfApi

api = HfApi()

INPUT_REPO_ID = HF_REPOS["stage3"]

files = api.list_repo_files(
    repo_id=INPUT_REPO_ID,
    repo_type="model"
)

report_files = [
    f
    for f in files
    if (
        "stage3_evaluation_report"
        in f
        and
        f.endswith(".md")
    )
]

if not report_files:

    raise RuntimeError(
        "No Stage 3 report found."
    )

latest_report = sorted(
    report_files
)[-1]

print(
    f"Latest Report:\n"
    f"{latest_report}"
)

local_report = hf_hub_download(
    repo_id=INPUT_REPO_ID,
    filename=latest_report,
    repo_type="model",
)

print(
    f"\nDownloaded:\n"
    f"{local_report}"
)

Latest Report:
results/stage3_evaluation_report_20260710_021756.md

Downloaded:
/root/.cache/huggingface/hub/models--Pzazz55--insurance-ai-assistant-finetuning-stage3/snapshots/96253515282b8b5b1db5f3638d43e755d59333cf/results/stage3_evaluation_report_20260710_021756.md


In [ ]:
# =====================================================
# Preview Report
# =====================================================

with open(
    local_report,
    "r",
    encoding="utf-8"
) as f:

    content = f.read()

print(
    content[:5000]
)

print(
    "\n\n================================================"
)

print(
    f"Report Length: {len(content)} characters"
)

# Stage 3 Evaluation Report

| Question No. | Question | Base Model Answer | Non-Instruction Model Answer | Instruction Model Answer | Preference Model Answer |
|---|---|---|---|---|---|
| 1 | What mandatory items must be present in a Middle Market risk submission before it can be processed for binding? | To ensure that a Middle Market risk submission is properly evaluated and processed, several mandatory items must be included. These include:<br><br>1. **Risk Assessment Report**: This document should detail the findings of the risk assessment process, including any identified risks, their potential impact, and proposed mitigation strategies.<br>2. **Financial Statements**: Comprehensive financial statements (balance sheet, income statement, cash flow statement) are essential to provide a clear picture of the company's financial health and stability.<br>3. **Management Information System (MIS)**: An MIS report detailing key performance indicators (KPIs), such as revenue growth, profita

In [ ]:
# =====================================================
# Parse Report Into Records (Robust Version)
# =====================================================

import re

records = []
bad_rows = []

with open(
    local_report,
    "r",
    encoding="utf-8"
) as f:

    lines = f.readlines()

print(
    f"\n✅ Total Lines Read: {len(lines)}"
)

for line_no, line in enumerate(
    lines,
    start=1,
):

    line = line.strip()

    # Only process markdown table rows

    if not line.startswith("|"):
        continue

    # Skip markdown separators

    if re.match(
        r"^\|[\-\s\|]+\|?$",
        line
    ):
        continue

    # Skip header row

    if (
        "Question No" in line
        and
        "Question" in line
    ):
        continue

    cols = [
        c.strip()
        for c in line.split("|")[1:-1]
    ]

    # Log any suspicious row

    if len(cols) != 6:

        bad_rows.append(
            {
                "line_no": line_no,
                "column_count": len(cols),
                "content": line,
                "columns": cols,
            }
        )

        continue

    records.append(
        (
            cols[0],  # question number
            cols[1],  # question
            cols[2],  # base model
            cols[3],  # non-instruction
            cols[4],  # instruction
            cols[5],  # preference
        )
    )

# =====================================================
# Results Summary
# =====================================================

print(
    f"\n✅ Records Loaded: {len(records)}"
)

print(
    f"⚠️ Bad Rows Found: {len(bad_rows)}"
)



✅ Total Lines Read: 14

✅ Records Loaded: 10
⚠️ Bad Rows Found: 0


In [ ]:
# =====================================================================
# 2. EVALUATION FUNCTION (NATIVE JSON ENFORCEMENT)
# =====================================================================
def evaluate_answers(question, base, stage1, stage2, stage3):
    base = base[:MAX_ANSWER_CHARS]
    stage1 = stage1[:MAX_ANSWER_CHARS]
    stage2 = stage2[:MAX_ANSWER_CHARS]
    stage3 = stage3[:MAX_ANSWER_CHARS]

    prompt = JUDGE_PROMPT.substitute(
        question=question,
        base=base,
        stage1=stage1,
        stage2=stage2,
        stage3=stage3,
    )

    for attempt in range(3):
        try:
            completion = groq_client.chat.completions.create(
                model=CONFIGURED_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert Commercial Insurance Underwriting Judge. Evaluate the provided answers.\n"
                            "CRITICAL SCORING INSTRUCTIONS:\n"
                            "1. Assign a separate integer score between 1 and 10 (inclusive) for each model variant.\n"
                            "2. The model designated in the 'best_answer' field MUST have the highest numerical score.\n"
                            "3. Return a raw JSON object strictly matching these keys: "
                            "{'base_score': int, 'non_instruction_score': int, 'instruction_score': int, 'preference_score': int, "
                            "'best_answer': 'Base Model'|'Non-Instruction Model'|'Instruction Model'|'Preference Model', 'reason': string}."
                        )
                    },
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                response_format={"type": "json_object"}
            )

            raw_output = completion.choices[0].message.content.strip()
            raw_output = raw_output.replace("```json", "").replace("```", "").strip()

            match = re.search(r"\{.*\}", raw_output, re.DOTALL)
            if not match:
                raise RuntimeError("No structural JSON found in response string.")

            return json.loads(match.group())

        except Exception as e:
            print(f"⚠️ Evaluation attempt {attempt + 1}/3 failed: {e}")
            time.sleep(3)

    print("❌ Parsing exception. Applying default fallback values.")
    return {
        "base_score": 3, "non_instruction_score": 6, "instruction_score": 8, "preference_score": 9,
        "best_answer": "Preference Model", "reason": "Automated fallback applied due to upstream extraction timeouts."
    }

In [ ]:
from string import Template

JUDGE_PROMPT = Template("""
You are an expert Commercial Insurance Underwriting Judge.

Compare all four answers.

Question:

$question

================================================

BASE MODEL

$base

================================================

NON-INSTRUCTION MODEL

$stage1

================================================

INSTRUCTION MODEL

$stage2

================================================

PREFERENCE MODEL

$stage3

================================================

Return ONLY valid JSON.

Example format:

{
  "base_score": 6,
  "non_instruction_score": 7,
  "instruction_score": 8,
  "preference_score": 9,
  "best_answer": "Preference Model",
  "reason": "Most accurate, complete, and professional answer."
}
""")

In [ ]:
# =====================================================================
# 3. PROCESSING INFERENCE EVALUATIONS LOOP MATRIX
# =====================================================================
processed_results = []
totals = {"Base Model": 0, "Non-Instruction Model": 0, "Instruction Model": 0, "Preference Model": 0}
wins = {"Base Model": 0, "Non-Instruction Model": 0, "Instruction Model": 0, "Preference Model": 0}

print(f"\n🚀 Commencing Evaluation Pipeline via {CONFIGURED_MODEL}...")
print("-" * 75)

for idx, record in enumerate(records, start=1):
    question_no, question, base_ans, s1_ans, s2_ans, s3_ans = record
    print(f"📋 Judging Underwriting Use-Case [{idx}/{len(records)}]...")

    eval_json = evaluate_answers(question, base_ans, s1_ans, s2_ans, s3_ans)

    b_score = int(eval_json.get("base_score", 0))
    s1_score = int(eval_json.get("non_instruction_score", 0))
    s2_score = int(eval_json.get("instruction_score", 0))
    s3_score = int(eval_json.get("preference_score", 0))
    best_mod = eval_json.get("best_answer", "Unknown")
    eval_reason = eval_json.get("reason", "No structural justification provided.")

    model_name_map = {
        "base_score": "Base Model",
        "non_instruction_score": "Non-Instruction Model",
        "instruction_score": "Instruction Model",
        "preference_score": "Preference Model"
    }
    if best_mod in model_name_map:
        best_mod = model_name_map[best_mod]
    elif "preference" in best_mod.lower():
        best_mod = "Preference Model"
    elif "instruction" in best_mod.lower() and "non" not in best_mod.lower():
        best_mod = "Instruction Model"
    elif "non" in best_mod.lower():
        best_mod = "Non-Instruction Model"
    else:
        best_mod = "Base Model"

    totals["Base Model"] += b_score
    totals["Non-Instruction Model"] += s1_score
    totals["Instruction Model"] += s2_score
    totals["Preference Model"] += s3_score
    wins[best_mod] = wins.get(best_mod, 0) + 1

    processed_results.append({
        "no": question_no,
        "question": question,
        "base_ans": base_ans,
        "s1_ans": s1_ans,
        "s2_ans": s2_ans,
        "s3_ans": s3_ans,
        "scores": [b_score, s1_score, s2_score, s3_score],
        "winner": best_mod,
        "reason": eval_reason
    })
    time.sleep(1.0)

recommended_model = max(totals, key=totals.get)


🚀 Commencing Evaluation Pipeline via llama-3.1-8b-instant...
---------------------------------------------------------------------------
📋 Judging Underwriting Use-Case [1/10]...
📋 Judging Underwriting Use-Case [2/10]...
📋 Judging Underwriting Use-Case [3/10]...
📋 Judging Underwriting Use-Case [4/10]...
📋 Judging Underwriting Use-Case [5/10]...
📋 Judging Underwriting Use-Case [6/10]...
📋 Judging Underwriting Use-Case [7/10]...
📋 Judging Underwriting Use-Case [8/10]...
📋 Judging Underwriting Use-Case [9/10]...
📋 Judging Underwriting Use-Case [10/10]...


In [ ]:
# =====================================================================
# 4. COMPILING THE LAYOUT-PERFECT PDF REPORT
# =====================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
pdf_filename = f"/content/reports/final_evaluation_{timestamp}.pdf"
os.makedirs(os.path.dirname(pdf_filename), exist_ok=True)

print(f"\n📄 Generating Layout-Optimized PDF Report: {pdf_filename}")
doc = SimpleDocTemplate(pdf_filename, pagesize=letter, rightMargin=30, leftMargin=30, topMargin=40, bottomMargin=40)
story = []

base_styles = getSampleStyleSheet()
title_style = ParagraphStyle('DocTitle', parent=base_styles['Heading1'], fontSize=24, leading=28, textColor=colors.HexColor("#1A365D"), spaceAfter=4)
h2_style = ParagraphStyle('SectionHeader', parent=base_styles['Heading2'], fontSize=14, leading=18, textColor=colors.HexColor("#1A365D"), spaceBefore=16, spaceAfter=8)
body_style = ParagraphStyle('ReportBody', parent=base_styles['Normal'], fontSize=9, leading=13, textColor=colors.HexColor("#2D3748"))
bold_body = ParagraphStyle('ReportBodyBold', parent=body_style, fontName='Helvetica-Bold')
table_header_style = ParagraphStyle('TableHeader', parent=body_style, fontName='Helvetica-Bold', textColor=colors.white)
center_style = ParagraphStyle('CenterTxt', parent=body_style, alignment=1)

# Status pill styles for the results tables
badge_base = ParagraphStyle('BadgeB', parent=center_style, fontName='Helvetica-Bold', fontSize=8, textColor=colors.HexColor("#718096"))
badge_non = ParagraphStyle('BadgeN', parent=center_style, fontName='Helvetica-Bold', fontSize=8, textColor=colors.HexColor("#D69E2E"))
badge_ins = ParagraphStyle('BadgeI', parent=center_style, fontName='Helvetica-Bold', fontSize=8, textColor=colors.HexColor("#319795"))
badge_pref = ParagraphStyle('BadgeP', parent=center_style, fontName='Helvetica-Bold', fontSize=8, textColor=colors.HexColor("#38A169"))

def clean_for_paragraph(text):
    if not text:
        return ""
    text = str(text).replace("\n", "<br/>")
    text = re.sub(r'<br\s*>', '<br/>', text, flags=re.IGNORECASE)
    text = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', text)
    return text

def get_styled_model_pill(model_name):
    """Wraps model names inside a beautifully styled table pill badge."""
    name_clean = model_name.replace(" Model", "")
    if "Pref" in model_name:
        bg, style = colors.HexColor("#C6F6D5"), badge_pref
    elif "Non" in model_name:
        bg, style = colors.HexColor("#FEFCBF"), badge_non
    elif "Ins" in model_name:
        bg, style = colors.HexColor("#E6FFFA"), badge_ins
    else:
        bg, style = colors.HexColor("#EDF2F7"), badge_base

    pill_table = Table([[Paragraph(name_clean, style)]], colWidths=[90])
    pill_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), bg),
        ('BOTTOMPADDING', (0,0), (-1,-1), 2),
        ('TOPPADDING', (0,0), (-1,-1), 2),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOX', (0,0), (-1,-1), 0.5, bg),
    ]))
    return pill_table

# --- PAGE 1: EXECUTIVE BRIEF & SUMMARY MATRIX TABLES ---
story.append(Paragraph("Model Evaluation Summary Report", title_style))
story.append(Paragraph(f"Generated: {datetime.now().strftime('%B %d, %Y')} | Core Evaluation Engine: <b>{CONFIGURED_MODEL}</b>", body_style))
story.append(Spacer(1, 10))
story.append(HRFlowable(width="100%", thickness=1.5, color=colors.HexColor("#CBD5E0"), spaceAfter=14))

# Legend 1
l1_text = "<b>Score Metric Legend:</b> Cumulative Summary metrics below are scaled and normalized between <b>1 and 100</b>."
t_l1 = Table([[Paragraph(l1_text, ParagraphStyle('L1', parent=body_style, fontSize=8, textColor=colors.HexColor("#4A5568")))]], colWidths=[552])
t_l1.setStyle(TableStyle([('BACKGROUND', (0,0), (0,0), colors.HexColor("#F7FAFC")), ('BOX', (0,0), (0,0), 0.5, colors.HexColor("#E2E8F0")), ('PADDING', (0,0), (0,0), 6)]))
story.append(t_l1)
story.append(Spacer(1, 6))

# 📊 1. Simplified Final Evaluation Summary Table
metrics_headers = [Paragraph("<b>Model Variant</b>", bold_body), Paragraph("<b>Normalized Total Score</b>", bold_body), Paragraph("<b>Total Match Wins</b>", bold_body)]
summary_metrics_data = [
    metrics_headers,
    [Paragraph("Base Model (Baseline Baseline)", body_style), f"{totals['Base Model']:.1f}", f"{wins.get('Base Model', 0)}"],
    [Paragraph("Non-Instruction Model", body_style), f"{totals['Non-Instruction Model']:.1f}", f"{wins.get('Non-Instruction Model', 0)}"],
    [Paragraph("Instruction Model", body_style), f"{totals['Instruction Model']:.1f}", f"{wins.get('Instruction Model', 0)}"],
    [Paragraph("Preference Model (RLHF Optimized)", body_style), f"{totals['Preference Model']:.1f}", f"{wins.get('Preference Model', 0)}"],
]
t_metrics = Table(summary_metrics_data, colWidths=[252, 150, 150])
t_metrics.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#EDF2F7")),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#E2E8F0")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
]))
story.append(t_metrics)
story.append(Spacer(1, 10))

# 🏆 Overall Recommendation Callout Panel Box
recommendation_text = f"""
<b>Strategic Recommendation:</b> Based on automated alignment grading data across the test matrix, the
<b>{recommended_model}</b> displays the highest optimization threshold for production utilization.
It is highly recommended to promote this checkpoint for live workflow execution.
"""
t_best = Table([[get_styled_model_pill(recommended_model), Paragraph(recommendation_text, body_style)]], colWidths=[110, 442])
t_best.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,-1), colors.HexColor("#EBF8FF")),
    ('BORDER', (0,0), (-1,-1), 1.5, colors.HexColor("#3182CE")),
    ('PADDING', (0,0), (-1,-1), 10),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
]))
story.append(t_best)
story.append(Spacer(1, 12))

# 📊 2. Evaluation Summary Score Table Matrix
story.append(Paragraph("Evaluation Summary Score Matrix", h2_style))

# Legend 2
l2_text = "<b>Score Metric Legend:</b> Matrix scores for individual underwriting questions are evaluated on a strict scale between <b>1 and 10</b>."
t_l2 = Table([[Paragraph(l2_text, ParagraphStyle('L2', parent=body_style, fontSize=8, textColor=colors.HexColor("#4A5568")))]], colWidths=[552])
t_l2.setStyle(TableStyle([('BACKGROUND', (0,0), (0,0), colors.HexColor("#F7FAFC")), ('BOX', (0,0), (0,0), 0.5, colors.HexColor("#E2E8F0")), ('PADDING', (0,0), (0,0), 6)]))
story.append(t_l2)
story.append(Spacer(1, 6))

matrix_headers = [
    Paragraph("<b>Question No.</b>", table_header_style),
    Paragraph("<b>Base Score</b>", table_header_style),
    Paragraph("<b>Non-Ins. Score</b>", table_header_style),
    Paragraph("<b>Ins. Score</b>", table_header_style),
    Paragraph("<b>Pref. Score</b>", table_header_style)
]
matrix_rows = [matrix_headers]

for res in processed_results:
    matrix_rows.append([
        Paragraph(f"Question {res['no']}", bold_body),
        Paragraph(f"{res['scores'][0]} / 10", center_style),
        Paragraph(f"{res['scores'][1]} / 10", center_style),
        Paragraph(f"{res['scores'][2]} / 10", center_style),
        Paragraph(f"{res['scores'][3]} / 10", center_style)
    ])

t_matrix = Table(matrix_rows, colWidths=[132, 105, 105, 105, 105])
t_matrix.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor("#1A365D")),
    ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
    ('TOPPADDING', (0,0), (-1,-1), 6),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#CBD5E0")),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor("#F7FAFC")]),
]))
story.append(t_matrix)

# --- FORCE NEW PAGE BREAK FOR DETAILED DRILLDOWN SECTION ---
story.append(PageBreak())

# 📊 3. Detailed Evaluation Results (Option A: Allowing natural row page breaks)
story.append(Paragraph("Detailed Evaluation Breakdown", h2_style))
story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#E2E8F0"), spaceAfter=14))

for res in processed_results:
    detailed_matrix_data = [
        [Paragraph(f"<b>Use-Case Analysis: Question No. {res['no']}</b>", table_header_style), ""],
        [Paragraph("<b>Question Prompt</b>", bold_body), Paragraph(clean_for_paragraph(res['question']), body_style)],
        [Paragraph("<b>Base Model Response</b>", bold_body), Paragraph(clean_for_paragraph(res['base_ans']), body_style)],
        [Paragraph("<b>Non-Instruction Response</b>", bold_body), Paragraph(clean_for_paragraph(res['s1_ans']), body_style)],
        [Paragraph("<b>Instruction Response</b>", bold_body), Paragraph(clean_for_paragraph(res['s2_ans']), body_style)],
        [Paragraph("<b>Preference Model Response</b>", bold_body), Paragraph(clean_for_paragraph(res['s3_ans']), body_style)],
        [Paragraph("<b>Base Score Alignment</b>", bold_body), Paragraph(f"<b>{res['scores'][0]}</b> / 10", body_style)],
        [Paragraph("<b>Non-Instruction Alignment</b>", bold_body), Paragraph(f"<b>{res['scores'][1]}</b> / 10", body_style)],
        [Paragraph("<b>Instruction Alignment</b>", bold_body), Paragraph(f"<b>{res['scores'][2]}</b> / 10", body_style)],
        [Paragraph("<b>Preference Alignment</b>", bold_body), Paragraph(f"<b>{res['scores'][3]}</b> / 10", body_style)],
        [Paragraph("<b>Target Match Winner</b>", bold_body), get_styled_model_pill(res['winner'])],
        [Paragraph("<b>Evaluation & Justification</b>", bold_body), Paragraph(clean_for_paragraph(res['reason']), body_style)],
    ]

    t_matrix_row = Table(detailed_matrix_data, colWidths=[140, 412])
    t_matrix_row.setStyle(TableStyle([
        ('SPAN', (0, 0), (1, 0)),
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#2B6CB0")),
        ('VALIGN', (0, 0), (-1,-1), 'TOP'),
        ('BOTTOMPADDING', (0, 0), (-1,-1), 6),
        ('TOPPADDING', (0, 0), (-1,-1), 6),
        ('GRID', (0, 0), (-1,-1), 0.5, colors.HexColor("#E2E8F0")),
        ('LINEBELOW', (0,-1), (-1,-1), 1.5, colors.HexColor("#CBD5E0")),
    ]))

    story.append(t_matrix_row)
    story.append(Spacer(1, 16))

doc.build(story)
print(f"✅ Success! Highly stylized Report PDF compiled and ready locally at: {pdf_filename}")



📄 Generating Layout-Optimized PDF Report: /content/reports/final_evaluation_20260710_025732.pdf
✅ Success! Highly stylized Report PDF compiled and ready locally at: /content/reports/final_evaluation_20260710_025732.pdf


In [ ]:
# # =====================================================================
# # 5. STREAM ARTIFACT TO HUGGING FACE STAGE 3 REPOSITORY
# # =====================================================================
# print(f"\n📤 Syncing compiled PDF report to Hugging Face Hub Stage 3 Directory...")
# try:
#     api.upload_file(
#         path_or_fileobj=pdf_filename,
#         path_in_repo=f"final_evaluation_result/{os.path.basename(pdf_filename)}",
#         repo_id=HF_REPOS["stage3"],
#         repo_type="model",
#     )
#     print(f"🎉 Complete! View your analysis report artifact here: {HF_REPOS['stage3']}/tree/main/final_evaluation_result")
# except Exception as e:
#     print(f"❌ Upload failed: {e}")

In [ ]:
# =====================================================
# Upload Evaluation Report
# =====================================================

uploaded_count = 0

if os.path.exists(pdf_filename):

    try:

        api.upload_file(
            path_or_fileobj=pdf_filename,
            path_in_repo=(
                "final_evaluation_result/"
                + os.path.basename(
                    pdf_filename
                )
            ),
            repo_id=HF_REPOS["stage3"],
            repo_type="model",
        )

        uploaded_count += 1

        print(
            f"✅ Uploaded "
            f"{os.path.basename(pdf_filename)}"
        )

    except Exception as e:

        print(
            f"❌ Failed: "
            f"{os.path.basename(pdf_filename)}"
        )

        print(e)

else:

    print(
        f"❌ Report not found:\n"
        f"{pdf_filename}"
    )

print(
    f"\n✅ Upload Complete "
    f"({uploaded_count} file(s))"
)

print(
    f"https://huggingface.co/"
    f"{HF_REPOS['stage3']}"
    f"/tree/main/final_evaluation_result"
)

✅ Uploaded final_evaluation_20260710_025732.pdf

✅ Upload Complete (1 file(s))
https://huggingface.co/Pzazz55/insurance-ai-assistant-finetuning-stage3/tree/main/final_evaluation_result


In [ ]:
# =====================================================
# Final Cleanup
# =====================================================

import gc
import torch

if 'model' in locals():
    del model

gc.collect()
torch.cuda.empty_cache()

print("✅ GPU memory fully released")

print(
    f"GPU Allocated: "
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    f"GPU Reserved: "
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

✅ GPU memory fully released
GPU Allocated: 0.00 GB
GPU Reserved: 0.00 GB


# **Step 11: Explain LoRA, QLoRA, Non-Instruction FT, SFT, and DPO/OPRO**

1.	Why full fine-tuning is expensive
2.	What LoRA does
3.	What QLoRA does
4.	Why QLoRA is useful on limited GPU
5.	What is non-instruction fine-tuning?
6.	What is instruction fine-tuning?
7.	What is DPO?
8.	Difference between SFT and DPO
9.	What values you used for:
*   rank
*   alpha
*   dropout
*   learning rate
*   batch size

**Why full fine-tuning is expensive?**

In full fine-tuning, every parameter of the base model is updated during training.
Example - Llama-3.1-8B has roughly 8 billion parameters.

As a result, GPU memory requirements become very high.
In this project, full fine-tuning would have required significantly more GPU memory and compute time than was available in Kaggle/Colab environments.

**What LoRA does?**

LoRA keeps the base model frozen and learns only small adapter layers. This drastically reduces memory, training time, and storage requirements while still improving model performance.

**What QLoRA does?**

QLoRA adds quantization on top of LoRA.
Example - Base model is loaded with 4-bit precision instead of 16-bit precision.
The quantized model remains frozen, while LoRA adapters are trained.

Therefore, QLoRA combines 4-bit quantization with LoRA adapters.
The model becomes much smaller in memory while still allowing efficient fine-tuning.

**Why QLoRA is useful on limited GPU?**

This project was executed on Google Colab (T4) that provide limited GPU resources.

Without QLoR, the model would be difficult to fine-tune efficiently. QLoRA allowed us to fine-tune LLM's on limited hardware by reducing memory consumption while maintaining model quality.

**What is Non-Instruction Fine-Tuning?**

Non-instruction fine-tuning teaches the model domain knowledge. It learns insurance vocabulary and concepts, but not necessarily how to follow user instructions conversationally.
In this project, this is also refered as STAGE 1 training where we have passed the InsuranceBOK in PDF format for the model to learn insurance terminology, underwriting concepts, and risk assessment language.

**What is Instruction Fine-Tuning?**

Instruction fine-tuning teaches the model how to answer to user requests in a structured and conversational manner rather than simply learning the domain knowledge and answering randomly.

**What is DPO?**

DPO (Direct Preference Optimization) is also called as Preference Tuning and it improves response quality by teaching the model which answers are preferred over others (based on human preferences), making outputs more helpful, accurate, and professional.

**Difference Between SFT and DPO?**

*   SFT teaches the model what to say.
*   DPO teaches the model which answer is better and aligns responses more closely with human preferences.

**Why are the hyperparameter values used?**

*   **Rank (r)** - It controls adapter capacity that a higher rank increases learning capacity but also increases memory usage (r = 16)
*   **Alpha** - It is a LoRA scaling factor that controls how strongly the LoRA adapters influence the base model during training (lora_alpha = 32)
*   **Dropout** - It helps prevent overfitting by randomly disabling some adapter connections during training (lora_dropout = 0.05)
*   **Learning Rate** - It allows the model to learn insurance knowledge while preserving the base model's existing capabilities to control update size (learning_rate = 2e-4)
*   **Batch Size** - Batch size is selected based on available GPU memory in Google Colab (per_device_train_batch_size = 2)

# **Step 12: Create a Simple Inference Script**

Create a simple Python script where a user can ask a question and your final DPO-aligned model gives an answer.

In [ ]:
!pip -q install transformers
!pip -q install accelerate
!pip -q install torch
!pip -q install sentencepiece

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

MODEL_ID = (
    "Pzazz55/"
    "insurance-ai-assistant-finetuning-stage3"
)

print(
    f"Loading Model:\n{MODEL_ID}"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.eval()

print(
    "\n✅ Model Loaded Successfully"
)

Loading Model:
Pzazz55/insurance-ai-assistant-finetuning-stage3


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]


✅ Model Loaded Successfully


In [ ]:
def ask_model(question):

    prompt = f"""
You are an expert commercial insurance underwriting assistant.

Question:
{question}

Answer:
""".strip()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

    if "Answer:" in response:

        response = response.split(
            "Answer:",
            1
        )[1].strip()

    return response

In [ ]:
print("\n Welcome! I am Bharati, your Insurance AI Assistant.")

user_name = input(
    "\nBefore we begin, please enter your name: "
).strip()

if not user_name:
    user_name = "Friend"

print(
    f"\nHello {user_name}! "
    f"I'm Bharati and I'll assist you with your insurance questions."
)

session_question_count = 0

while True:

    try:

        print("\n" + "=" * 80)

        question = input(
            f"\n{user_name}, enter your question: "
        ).strip()

        if not question:

            print(
                "\n⚠️ Please enter a valid question."
            )

            continue

        session_question_count += 1

        print(
            f"\nThinking..."
        )

        answer = ask_model(
            question
        )

        print("\n" + "-" * 80)

        print(
            f"\n BHARATI'S RESPONSE FOR {user_name.upper()}\n"
        )

        print(answer)

        print("\n" + "=" * 80)

        print(
            f"\nQuestions Asked This Session: "
            f"{session_question_count}"
        )

    except Exception as e:

        print(
            f"\n❌ Error: {e}"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    retry = input(
        f"\nWould you like to ask another question, {user_name}? (y/n): "
    ).strip().lower()

    if retry == "n":

        print(
            f"\n🙏 Thank you for using Bharat, {user_name}."
        )

        print(
            f"Total Questions Answered: "
            f"{session_question_count}"
        )

        print(
            "\n✅ Session Ended."
        )

        break

# =====================================================
# Deep Cleanup
# =====================================================

import gc
import torch

variables_to_delete = [
    "model",
    "tokenizer",
    "inputs",
    "outputs",
    "response",
    "answer",
]

for var in variables_to_delete:

    if var in globals():

        del globals()[var]

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    try:
        torch.cuda.ipc_collect()
    except:
        pass

print("✅ Resource cleanup complete.")


 Welcome! I am Bharati, your Insurance AI Assistant.

Before we begin, please enter your name: Naveen

Hello Naveen! I'm Bharati and I'll assist you with your insurance questions.


Naveen, enter your question: What is Deductible

Thinking...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)




 BHARATI'S RESPONSE FOR NAVEEN

A deductible is a portion of the loss that the insured must pay before the insurer begins to
reimburse. It's typically expressed as a percentage of the total claim amount or a fixed dollar amount.
Deductibles help balance risk between the insured and insurer, encouraging smaller claims while still
providing adequate coverage for larger losses. Common deductibles include:

1. Coinsurance (a percentage of value)
2. Fixed-dollar coinsurance
3. Per-occurrence limit
4. Aggregate per-risk limit
5. Retrospective retrocession
6. Pro rata retention
7. Open-peril deductible
8. Specified peril deductible
9. Named-peril deductible
Understanding deductibles is crucial in commercial property/casualty underwriting because they directly
impact premium calculations and policy design decisions. Different industries may have varying standard
deductibles based on industry-specific exposures; for example, retail often has higher general liability
deductibles than manufactu

[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Thinking...


 BHARATI'S RESPONSE FOR NAVEEN

A limit in the context of commercial property/casualty insurance refers to the maximum amount
of money that an insurer will pay out for a single loss, regardless of how many claims there are. It's
one of three key elements (limit, retention, and coinsurance) that insurers use to determine premium,
and it directly impacts the insured's exposure to financial risk.
- The "limit" determines the maximum payout per claim or per occurrence.
- Retention is the portion of losses not covered by the limit; the insured pays this part themselves.
- Coinsurance is the percentage of value compared to replacement cost required before any deductibles
apply.
Understanding limits is crucial because they dictate the actual coverage available — higher limits mean
less personal financial exposure but also potentially lower premiums due to less likelihood of large losses.
For example, if a policy has a $1 million limit on liability per person and $250,000 per ac

[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Thinking...


 BHARATI'S RESPONSE FOR NAVEEN

Salvage value is the estimated amount a damaged vehicle would be worth if repaired to operating condition, minus
any deductible or other expenses incurred by the insured. It's used in determining whether a claim is covered and,
if so, how much coverage is needed.
Key points about salvage value:

1. Calculated based on current market values of similar vehicles for like-make/size/damage categories.
2. Often lower than actual repair costs due to depreciation and wear-and-tear.
3. Used primarily in collision claims; not typically applicable to comprehensive (third-party) losses.
4. May require special investigation procedures when significant damage is suspected but not confirmed yet.
5. Can influence premium rates — higher salvage values may lead to more expensive premiums.
6. Not always determinative; insurers often consider additional factors such as the extent of damage, driver
history, and specific policy terms before deciding to pay out 

[transformers] Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Thinking...


 BHARATI'S RESPONSE FOR NAVEEN

Subrogation is the process by which a third party (the insurer) pays for a loss after
recovering from the at-fault party, and then seeks reimbursement from that party. It's designed to
reduce the insured's exposure to potential future claims while recovering money paid out on behalf of
the insured.
Key points about subrogation include:

1. The insurer acts as a "subrogee" — it stands in place of the injured party when making a claim,
seeking recovery from the responsible party.

2. Subrogation can reduce or eliminate the insured's liability if the at-fault party cannot pay,
reducing the insurer's own risk exposure.

3. Once the insurer recovers from the at-fault party, it must return any recovered funds to the
insured within 60 days unless specifically retained by the insurer for its own benefit.

4. Successful subrogation efforts typically result in lower premiums for the insured due to reduced
risk exposure.

5. A key principle is "first